# Swin Feature Extractor -> SVM Classifier

In [ ]:
# Import Library
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import timm
from PIL import Image
from tqdm import tqdm
from torchvision import datasets, transforms, models

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

warnings.filterwarnings("ignore")

In [ ]:
# Setup 
df = pd.read_csv('dataset_filtered.csv')
df['use_no_bg'] = df['use_no_bg'].astype(bool)

label_to_idx = {
    label: idx
    for idx, label in enumerate(sorted(df["label"].unique()))
}

idx_to_label = {
    idx: label
    for label, idx in label_to_idx.items()
}

class_names = list(label_to_idx.keys())

df.head()

In [ ]:
# Configuration
class Config:

    # Random Seed
    SEED = 42

    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_WORKERS = 0

    # Dataset
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    NUM_CLASSES = 3

    # Data Path
    TRAIN_DIR = "train"
    TEST_DIR = "test"

    # Model
    SWIN_MODEL_NAME = "swin_t"
    SWIN_WEIGHT_PATH = "swin_t.pth"

    # Output
    SUBMISSION_PATH = "submission.csv"
    SVM_MODEL_PATH = "svm_classifier.pth"

print(f"Device : {Config.DEVICE}")

In [ ]:
# Random Seed
def set_seed(seed: int):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(Config.SEED)

In [ ]:
class MixedDataset(Dataset):
    """
    Dataset bakal memilih acak gambar yang di-augmentasi berikut:
    - Gambar asli + augmentasi (sedikit lebih agresif)
    - Gambar no-background + augmentasi (jika use_no_bg=True)
    """
    def __init__(self, dataframe, heavy_transform, light_transform):
        self.df = dataframe.reset_index(drop=True)
        self.heavy = heavy_transform
        self.light = light_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = label_to_idx[row['label']]

        if row['use_no_bg'] and random.random() < 0.5:
            img = Image.open(row['final_path']).convert('RGB')
            img = self.light(img)
        else:
            img = Image.open(row['file_path']).convert('RGB')
            img = self.heavy(img)
        return img, label

In [ ]:
class EvalDataset(Dataset):
    """
    Dataset untuk validasi (nanti di split)
    """
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['file_path']).convert('RGB')
        img = self.transform(img)
        return img, label_to_idx[row['label']]

In [ ]:
# Heavy Augmentation
htrain = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3,
        hue=0.1
    ),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),
])

# Light Augmentation
ltrain = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.3, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),
])

# Evaluation
etransform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Dataset & Train Validation Split
full_dataset = MixedDataset(df, htrain, ltrain)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(Config.SEED)

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator
)

df_val = df.iloc[val_dataset.indices].reset_index(drop=True)

val_eval_dataset = EvalDataset(
    df_val,
    etransform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_eval_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [ ]:
# Build Swin Model
def build_model(num_classes):
    """
    Build Swin Transformer model.
    """

    model = models.swin_t(weights=None)

    model.head = nn.Linear(
        model.head.in_features,
        num_classes
    )

    return model

def load_model(weight_path, num_classes, device):
    """
    Load pretrained Swin Transformer.
    """

    model = build_model(num_classes)

    state_dict = torch.load(
        weight_path,
        map_location=device
    )

    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    for param in model.parameters():
        param.requires_grad = False

    return model

In [ ]:
swin_model = load_model(
    Config.SWIN_WEIGHT_PATH,
    Config.NUM_CLASSES,
    Config.DEVICE
)

print("Swin model loaded.")

In [ ]:
# Swin Feature Extractor
class SwinFeatureExtractor:

    def __init__(self, model, device):
        self.model = model
        self.device = device

    def extract(self, dataloader, desc="Extracting Features"):
        """
        Extract feature vectors from images.

        Returns:
            features (numpy.ndarray)
            labels (numpy.ndarray)
        """

        self.model.eval()

        features = []
        labels = []

        with torch.no_grad():

            for images, targets in tqdm(
                dataloader,
                desc,
                unit="batch"
            ):

                images = images.to(self.device)

                x = self.model.features(images)
                x = self.model.norm(x)

                x = x.permute(0, 3, 1, 2)
                x = torch.nn.functional.adaptive_avg_pool2d(x, (1, 1))
                x = torch.flatten(x, 1)

                features.append(x.cpu().numpy())
                labels.append(targets.numpy())

        features = np.concatenate(features, axis=0)
        labels = np.concatenate(labels, axis=0)

        return features, labels

    def extract_test(self, dataloader, desc):

        self.model.eval()
    
        features = []
        ids = []
    
        with torch.no_grad():
    
            for images, img_ids in tqdm(
                dataloader,
                desc=desc,
                unit="batch"
            ):
    
                images = images.to(self.device)
    
                x = self.model.features(images)
                x = self.model.norm(x)
    
                x = x.permute(0, 3, 1, 2)
                x = torch.nn.functional.adaptive_avg_pool2d(x, (1, 1))
                x = torch.flatten(x, 1)
    
                features.append(x.cpu().numpy())
                ids.extend(img_ids)
    
        features = np.concatenate(features, axis=0)
    
        return features, ids

In [ ]:
# SVM Classifier
class SVMClassifier:

    def __init__(self):
        self.model = SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            random_state=Config.SEED
        )

    def fit(self, X, y):
        self.model.fit(X, y)

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def save(self, path):
        torch.save(self.model, path)

    def load(self, path):
        self.model = torch.load(path)

In [ ]:
# Swin Feature + SVM Pipeline
class SwinFeatureSVMPipeline:

    def __init__(self, extractor, classifier):
        self.extractor = extractor
        self.classifier = classifier

        # Feature
        self.X_train = None
        self.y_train = None
        self.X_val = None
        self.y_val = None
        self.X_test = None
        self.test_ids = None

        # Prediction
        self.y_pred = None
        self.test_pred = None

        # Metric
        self.accuracy = None

    def train(self, train_loader):

        self.X_train, self.y_train = self.extractor.extract(
            train_loader,
            desc="Train Features"
        )

        self.classifier.fit(
            self.X_train,
            self.y_train
        )

        self.save(Config.SVM_MODEL_PATH)

        print(f"SVM model saved to {Config.SVM_MODEL_PATH}")

    def evaluate(self, val_loader):

        self.X_val, self.y_val = self.extractor.extract(
            val_loader,
            desc="Validation Features"
        )

        self.y_pred = self.classifier.predict(self.X_val)

        self.accuracy = accuracy_score(
            self.y_val,
            self.y_pred
        )

        print(f"\nValidation Accuracy : {self.accuracy:.4f}\n")
        print(classification_report(
            self.y_val,
            self.y_pred
        ))

    def predict(self, test_loader):

        self.X_test, self.test_ids = self.extractor.extract_test(
            test_loader,
            desc="Test Features"
        )
    
        self.test_pred = self.classifier.predict(
            self.X_test
        )
    
        return self.test_pred

    def save(self, path):
        self.classifier.save(path)

In [ ]:
extractor = SwinFeatureExtractor(
    model=swin_model,
    device=Config.DEVICE
)

svm = SVMClassifier()

pipeline = SwinFeatureSVMPipeline(
    extractor,
    svm
)

In [ ]:
pipeline.train(train_loader)

In [ ]:
pipeline.evaluate(val_loader)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    pipeline.y_val,
    pipeline.y_pred,
    display_labels=class_names,
    cmap="Blues"
)

plt.title("Swin Feature Extractor + SVM Confusion Matrix")
plt.show()

In [ ]:
class TestDataset(Dataset):
    def __init__(self, ids, img_dir, transform):
        self.ids = ids
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]

        img = Image.open(
            os.path.join(self.img_dir, img_id)
        ).convert("RGB")

        img = self.transform(img)

        return img, img_id

In [ ]:
# Test DataLoader
submission = pd.read_csv("submission.csv")

test_ids = submission["id"].astype(str) + ".jpg"

test_dataset = TestDataset(
    test_ids,
    Config.TEST_DIR,
    etransform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"Test samples : {len(test_dataset)}")

In [ ]:
pipeline.predict(test_loader)

In [ ]:
submission = pd.DataFrame({
    "id": [
        os.path.splitext(img_id)[0]
        for img_id in pipeline.test_ids
    ],
    "label": [
        idx_to_label[pred]
        for pred in pipeline.test_pred
    ]
})

submission.to_csv(
    "submission_svm.csv",
    index=False
)

submission.head()